In [3]:
# ============================================================
#  DecodeLabs | Industrial Training Kit | Batch 2026
#  Project 2: Data Classification Using AI
#  Algorithm: K-Nearest Neighbors (KNN) on the Iris Dataset
# ============================================================
 
# ── 1. IMPORTS ───────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
 
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
)

In [6]:
# ── 2. LOAD & EXPLORE THE DATASET ────────────────────────────
iris = load_iris()
 
X = iris.data        # Features: sepal length, sepal width, petal length, petal width
y = iris.target      # Labels:   0 = Setosa, 1 = Versicolor, 2 = Virginica
 

print(f"Dataset Shape   : {X.shape}  (samples × features)")
print(f" Classes         : {iris.target_names.tolist()}")
print(f"Samples/class   : {np.bincount(y).tolist()}  (balanced)")
print(f"Feature names   : {iris.feature_names}")

Dataset Shape   : (150, 4)  (samples × features)
 Classes         : ['setosa', 'versicolor', 'virginica']
Samples/class   : [50, 50, 50]  (balanced)
Feature names   : ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']


In [7]:
# ── 3. FEATURE SCALING (StandardScaler) ──────────────────────
#  KNN is distance-based — unscaled features bias results.
#  StandardScaler transforms each feature to mean=0, variance=1.
 
scaler = StandardScaler()
 
# ── 4. TRAIN-TEST SPLIT (80 / 20) ────────────────────────────
#  shuffle=True removes order bias before splitting.
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, shuffle=True, stratify=y
)
 
# Fit scaler ONLY on training data, then transform both sets
# (prevents data leakage from test set into training)
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)
 
print(f"\nTrain samples   : {X_train.shape[0]}")
print(f"Test  samples   : {X_test.shape[0]}")


Train samples   : 120
Test  samples   : 30


In [14]:
# ── 5. FIND OPTIMAL K (Elbow Method) ─────────────────────────
error_rates = []
k_range = range(3, 21)
 
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    preds = knn.predict(X_test)
    error_rates.append(1 - accuracy_score(y_test, preds))
 
optimal_k = k_range[np.argmin(error_rates)]
print(f"\nOptimal K       : {optimal_k}  (lowest error rate)")
 
# Plot the elbow curve
plt.figure(figsize=(9, 4))
plt.plot(k_range, error_rates, marker="o", color="#1a3a5c", linewidth=2)
plt.axvline(optimal_k, color="#e8490f", linestyle="--", label=f"Optimal K = {optimal_k}")
plt.scatter([optimal_k], [error_rates[optimal_k - 1]],
            color="#e8490f", s=120, zorder=5)
plt.title("Tuning the Engine: Choosing K (Elbow Method)", fontsize=13, fontweight="bold")
plt.xlabel("K Value")
plt.ylabel("Error Rate")
plt.legend()
plt.tight_layout()
plt.savefig("elbow_curve.png", dpi=150)
plt.close()
print("Saved           : elbow_curve.png")


Optimal K       : 7  (lowest error rate)
Saved           : elbow_curve.png


In [10]:
# ── 6. TRAIN THE FINAL MODEL ──────────────────────────────────
model = KNeighborsClassifier(n_neighbors=optimal_k)
model.fit(X_train, y_train)          # FIT  — memorise the map
predictions = model.predict(X_test)  # PREDICT — apply logic
 
# ── 7. OUTPUT VALIDATION ──────────────────────────────────────
acc    = accuracy_score(y_test, predictions)
f1     = f1_score(y_test, predictions, average="weighted")
cm     = confusion_matrix(y_test, predictions)
report = classification_report(y_test, predictions,
                               target_names=iris.target_names)
 
print("\n" + "─" * 55)
print("  OUTPUT VALIDATION")
print("─" * 55)
print(f"Accuracy Score  : {acc:.4f}  ({acc*100:.2f}%)")
print(f"F1 Score (wtd)  : {f1:.4f}")
print("\nClassification Report:\n")
print(report)


───────────────────────────────────────────────────────
  OUTPUT VALIDATION
───────────────────────────────────────────────────────
Accuracy Score  : 0.9667  (96.67%)
F1 Score (wtd)  : 0.9666

Classification Report:

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.91      1.00      0.95        10
   virginica       1.00      0.90      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [11]:
# ── 8. CONFUSION MATRIX HEATMAP ──────────────────────────────
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=iris.target_names,
    yticklabels=iris.target_names,
    linewidths=0.5,
)
plt.title("The Diagnostic Tool: Confusion Matrix", fontsize=13, fontweight="bold")
plt.ylabel("Actual Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.close()
print("Saved           : confusion_matrix.png")

Saved           : confusion_matrix.png


In [12]:
# ── 9. FEATURE IMPORTANCE (via variance across classes) ───────
#  KNN has no built-in feature importance, but we can visualise
#  how well each feature separates the three classes.
 
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
colors = ["#1a3a5c", "#e8490f", "#4caf50"]
 
for i, (ax, feat) in enumerate(zip(axes, iris.feature_names)):
    for cls_idx, cls_name in enumerate(iris.target_names):
        ax.hist(X[y == cls_idx, i], bins=12, alpha=0.6,
                color=colors[cls_idx], label=cls_name)
    ax.set_title(feat, fontsize=9)
    ax.set_xlabel("cm")
    if i == 0:
        ax.set_ylabel("Count")
    ax.legend(fontsize=7)
 
fig.suptitle("Feature Distributions by Iris Class", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("feature_distributions.png", dpi=150)
plt.close()
print("Saved           : feature_distributions.png")

Saved           : feature_distributions.png


In [13]:
# ── 10. PREDICT A NEW SAMPLE ─────────────────────────────────
print("\n" + "─" * 55)
print("  LIVE PREDICTION DEMO")
print("─" * 55)
 
new_sample = np.array([[5.1, 3.5, 1.4, 0.2]])   # typical Setosa
new_scaled  = scaler.transform(new_sample)
prediction  = model.predict(new_scaled)
probability = model.predict_proba(new_scaled)
 
print(f"Input features     : sepal_len=5.1, sepal_wid=3.5, "
      f"petal_len=1.4, petal_wid=0.2")
print(f"Predicted class    : {iris.target_names[prediction[0]].upper()}")
print(f"Class probabilities: ", end="")
for name, prob in zip(iris.target_names, probability[0]):
    print(f"{name}={prob:.2f}", end="  ")
print()


───────────────────────────────────────────────────────
  LIVE PREDICTION DEMO
───────────────────────────────────────────────────────
Input features     : sepal_len=5.1, sepal_wid=3.5, petal_len=1.4, petal_wid=0.2
Predicted class    : SETOSA
Class probabilities: setosa=1.00  versicolor=0.00  virginica=0.00  
